# Temporal Calibration Pilot — Results Viewer

**No GPU required.** This notebook loads the pre-computed outputs from the 14B run and displays them.

**Model:** DeepSeek-R1-Distill-Qwen-14B (fp16) | **N =** 50 questions × 2 modes = 100 calls

Click **Run All** — takes under 30 seconds.

In [ ]:
# Locate the results files (works on Kaggle after cloning, or locally)
import subprocess, shutil, sys
from pathlib import Path

REPO_URL = "https://github.com/kumarswamyg2005/temporal-calib-pilot.git"
CLONE_DIR = Path("/kaggle/working/temporal-calib-pilot")
ON_KAGGLE = Path("/kaggle/working").is_dir()

if ON_KAGGLE:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, str(CLONE_DIR)],
        check=True, capture_output=True,
    )
    NOTEBOOKS_DIR = CLONE_DIR / "notebooks"
else:
    NOTEBOOKS_DIR = Path("__file__").resolve().parent if "__file__" in dir() else Path(".").resolve()
    # If run from repo root
    if (NOTEBOOKS_DIR / "notebooks").exists():
        NOTEBOOKS_DIR = NOTEBOOKS_DIR / "notebooks"

SLUG = "deepseek-ai_DeepSeek-R1-Distill-Qwen-14B"
SUMMARY_CSV = NOTEBOOKS_DIR / f"summary_table__{SLUG}.csv"
RESULTS_CSV = NOTEBOOKS_DIR / f"results__{SLUG}.csv"
CHART_PNG   = NOTEBOOKS_DIR / f"pilot_chart__{SLUG}.png"

for f in [SUMMARY_CSV, RESULTS_CSV, CHART_PNG]:
    status = "✓" if f.exists() else "MISSING"
    print(f"{status}  {f.name}")

## 1. Headline Chart

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(CHART_PNG), width=700))

## 2. Per-Bucket Summary Table

In [ ]:
import pandas as pd

summary = pd.read_csv(SUMMARY_CSV)
summary["accuracy_%"]   = (summary["accuracy"] * 100).round(0).astype(int).astype(str) + "%"
summary["confidence"]   = summary["mean_confidence"].round(1)
summary["gap"]          = summary["overconfidence_gap"].round(1)
summary["ece"]          = summary["ece"].round(3)

display(
    summary[["bucket", "distance_months", "mode", "n", "accuracy_%", "confidence", "gap", "ece"]]
    .rename(columns={
        "distance_months": "months_to_cutoff",
        "accuracy_%": "accuracy",
        "confidence": "mean_conf",
        "gap": "overconf_gap",
    })
    .style.set_caption(
        "Overconfidence gap = mean_confidence − accuracy×100. "
        "Positive = overconfident · Negative = underconfident."
    )
    .applymap(lambda v: "color: red" if isinstance(v, float) and v > 5 else
                        ("color: steelblue" if isinstance(v, float) and v < -20 else ""),
              subset=["overconf_gap"])
)

## 3. Overall Stats & Parse Quality

In [ ]:
results = pd.read_csv(RESULTS_CSV)

overall = (
    results.groupby("mode")
    .agg(
        n=("id", "count"),
        accuracy=("is_correct", "mean"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        parse_ok=("parse_ok", "mean"),
    )
    .round(3)
    .reset_index()
)
overall["accuracy"] = (overall["accuracy"] * 100).round(1).astype(str) + "%"
overall["parse_ok"] = (overall["parse_ok"] * 100).round(1).astype(str) + "%"
display(overall)

## 4. Verdict

In [ ]:
def _gap(bucket, mode):
    r = summary[(summary["bucket"] == bucket) & (summary["mode"] == mode)]
    return float(r["overconfidence_gap"].iloc[0]) if len(r) else float("nan")

g_on_b5,  g_off_b5  = _gap("B5", "thinking_on"),  _gap("B5", "thinking_off")
g_on_b1,  g_off_b1  = _gap("B1", "thinking_on"),  _gap("B1", "thinking_off")

mode_diff_near = g_on_b5  - g_off_b5
on_drift       = g_on_b5  - g_on_b1
strength       = min(mode_diff_near, on_drift)

if strength >= 15:
    verdict = "Strong signal — Thinking ON is clearly worse-calibrated near the cutoff. Scale up."
elif strength >= 5:
    verdict = "Mild signal — Thinking ON shows a temporal calibration gradient. Scale up to confirm."
else:
    verdict = "No clear signal. Pivot or rethink hypothesis."

acc_on  = results[results["mode"] == "thinking_on"]["is_correct"].mean() * 100
acc_off = results[results["mode"] == "thinking_off"]["is_correct"].mean() * 100

print("=" * 60)
print("PILOT STUDY — DeepSeek-R1-Distill-Qwen-14B (fp16)")
print("=" * 60)
print(f"Questions evaluated : 50 × 2 modes = 100 calls")
print(f"Thinking ON accuracy : {acc_on:.0f}%")
print(f"Thinking OFF accuracy: {acc_off:.0f}%")
print()
print(f"Overconfidence gap at B5 (1 month from cutoff):")
print(f"  Thinking ON  = {g_on_b5:.1f} pp")
print(f"  Thinking OFF = {g_off_b5:.1f} pp")
print(f"  ON − OFF     = {mode_diff_near:+.1f} pp")
print()
print(f"ON drift B1→B5 (24mo → 1mo): {on_drift:+.1f} pp")
print(f"Signal strength (min of above): {strength:.1f}")
print()
print(f"VERDICT: {verdict}")
print()
print("Note: Both conditions are UNDERCONFIDENT overall (gap < 0).")
print("The model expresses low numeric confidence but Thinking ON")
print("is consistently less underconfident, especially near the cutoff.")